In [1]:
import sqlite3
import pandas as pd

# Connect to Chinook database
conn = sqlite3.connect('chinook.db')

print("Connected to Chinook database")

Connected to Chinook database


In [2]:
# Configure pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 30)

In [5]:
SCHEMA REFERENCE
Key Tables andColumns
tracks

TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer
Milliseconds, Bytes, UnitPrice

albums

AlbumId, Title, ArtistId

artists

ArtistId, Name

customers

CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId

invoices

InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total

invoice_items

InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity

genres

GenreId, Name

media_types

MediaTypeId, Name

employees

EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email

SyntaxError: invalid syntax (1543158634.py, line 1)

In [5]:
# Problem 1: Rank customers by total spending

query = """
        SELECT 
            Customer.FirstName AS Name,
            SUM(Invoice.Total) AS TotalSpending,
            RANK() OVER (ORDER BY SUM(Invoice.Total) DESC) AS SpendingRank
        FROM Customer
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
        GROUP BY Customer.CustomerId, Customer.FirstName;
        """

result = pd.read_sql_query(query, conn)
print(result)

         Name  TotalSpending  SpendingRank
0      Helena          49.62             1
1     Richard          47.62             2
2        Luis          46.62             3
3    Ladislav          45.62             4
4        Hugh          45.62             4
5       Frank          43.62             6
6       Julia          43.62             6
7        Fynn          43.62             6
8      Astrid          42.62             9
9      Victor          42.62             9
10      Terhi          41.62            11
11  František          40.62            12
12   Isabelle          40.62            12
13   Johannes          40.62            12
14       Luís          39.62            15
15   François          39.62            15
16      Bjørn          39.62            15
17       Jack          39.62            15
18        Dan          39.62            15
19    Heather          39.62            15
20       João          39.62            15
21      Wyatt          39.62            15
22   Jennif

In [7]:
# Problem 2: Rank Customers within each country

query = """
            SELECT 
                Customer.FirstName AS Customer,
                SUM(Invoice.Total) AS Total_Spending,
                Customer.Country,
                RANK() OVER (PARTITION BY Customer.Country ORDER BY SUM(Invoice.Total) DESC) AS CountryRank
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
            GROUP BY Customer.CustomerId, Customer.FirstName, Customer.Country;
        
        """

result = pd.read_sql_query(query, conn)
print(result)

     Customer  Total_Spending         Country  CountryRank
0       Diego           37.62       Argentina            1
1        Mark           37.62       Australia            1
2      Astrid           42.62         Austria            1
3        Daan           37.62         Belgium            1
4        Luís           39.62          Brazil            1
5     Eduardo           37.62          Brazil            2
6   Alexandre           37.62          Brazil            2
7     Roberto           37.62          Brazil            2
8    Fernanda           37.62          Brazil            2
9    François           39.62          Canada            1
10   Jennifer           38.62          Canada            2
11       Mark           37.62          Canada            3
12     Robert           37.62          Canada            3
13     Edward           37.62          Canada            3
14     Martha           37.62          Canada            3
15      Aaron           37.62          Canada           

In [16]:
# Problem 3: Running total of revenue over time

query = """
        SELECT
            Invoice.InvoiceDate,
            SUM(Invoice.Total) AS DailyRevenue,
            SUM(SUM(Invoice.Total)) OVER(ORDER BY Invoice.InvoiceDate) AS Running_Total
        FROM Invoice
        GROUP BY Invoice.InvoiceDate;
        
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)

             InvoiceDate  DailyRevenue  Running_Total
0    2021-01-01 00:00:00          1.98           1.98
1    2021-01-02 00:00:00          3.96           5.94
2    2021-01-03 00:00:00          5.94          11.88
3    2021-01-06 00:00:00          8.91          20.79
4    2021-01-11 00:00:00         13.86          34.65
..                   ...           ...            ...
349  2025-12-05 00:00:00          3.96        2297.90
350  2025-12-06 00:00:00          5.94        2303.84
351  2025-12-09 00:00:00          8.91        2312.75
352  2025-12-14 00:00:00         13.86        2326.61
353  2025-12-22 00:00:00          1.99        2328.60

[354 rows x 3 columns]


In [22]:
# Problem 4: Compare each invoice amount to the customer's average invoice

query = """
        SELECT Customer.FirstName AS Customer,
            Invoice.Total AS Invoice_Amount,
            AVG(Invoice.Total) OVER(PARTITION BY Customer.CustomerId) as CustomerAvg,
            Invoice.Total - AVG(Invoice.Total) OVER(PARTITION BY Customer.CustomerId) AS Difference
        FROM Customer
        JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId;
        
        
        
        
            
        
        """

result = pd.read_sql_query(query, conn)
print(result)

    Customer  Invoice_Amount  CustomerAvg  Difference
0       Luís            3.98     5.660000   -1.680000
1       Luís            3.96     5.660000   -1.700000
2       Luís            5.94     5.660000    0.280000
3       Luís            0.99     5.660000   -4.670000
4       Luís            1.98     5.660000   -3.680000
..       ...             ...          ...         ...
407     Puja            5.94     6.106667   -0.166667
408     Puja            1.99     6.106667   -4.116667
409     Puja            1.98     6.106667   -4.126667
410     Puja           13.86     6.106667    7.753333
411     Puja            8.91     6.106667    2.803333

[412 rows x 4 columns]


In [6]:
# Problem 5: Find the most expensive tracks in each genre


query = """
            SELECT 
            Track.Name,
            Genre.Name,
            Track.UnitPrice,
            MAX(Track.UnitPrice) AS Most_Expensive
            FROM Track
            JOIN Genre ON Track.GenreId = Genre.GenreId
            GROUP BY Genre.GenreId, Genre.Name;           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                             Name                Name  UnitPrice  \
0   For Those About To Rock (W...                Rock       0.99   
1                      Desafinado                Jazz       0.99   
2                   Enter Sandman               Metal       0.99   
3              Your Time Has Come  Alternative & Punk       0.99   
4                           Money       Rock And Roll       0.99   
5      First Time I Met The Blues               Blues       0.99   
6              Jorge Da Capadócia               Latin       0.99   
7                        Girassol              Reggae       0.99   
8   Dig-Dig, Lambe-Lambe (Ao V...                 Pop       0.99   
9                    Vai-Vai 2001          Soundtrack       0.99   
10                Samba Da Bênção          Bossa Nova       0.99   
11                         My Way      Easy Listening       0.99   
12                 Wildest Dreams         Heavy Metal       0.99   
13           Please Please Please            R&B

In [12]:
# Problem 6: Find the most expensive tracks in each genre


query = """
            SELECT
    t.Name,
    t.UnitPrice,
    t.Genre,
    t.PercentileRank
FROM (
    SELECT
        Track.Name AS Name,
        Track.UnitPrice AS UnitPrice,
        Genre.Name AS Genre,
        PERCENT_RANK() OVER(
            PARTITION BY Genre.GenreId ORDER BY Track.UnitPrice DESC
        ) AS PercentileRank
    FROM Track
    JOIN Genre ON Track.GenreId = Genre.GenreId
) AS t
WHERE t.PercentileRank <= 0.1;
                       
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                               Name  UnitPrice      Genre  PercentileRank
0     For Those About To Rock (W...       0.99       Rock             0.0
1                 Balls to the Wall       0.99       Rock             0.0
2                   Fast As a Shark       0.99       Rock             0.0
3                 Restless and Wild       0.99       Rock             0.0
4              Princess of the Dawn       0.99       Rock             0.0
...                             ...        ...        ...             ...
3498  Pini Di Roma (Pinien Von R...       0.99  Classical             0.0
3499  String Quartet No. 12 in C...       0.99  Classical             0.0
3500  L'orfeo, Act 3, Sinfonia (...       0.99  Classical             0.0
3501  Quintet for Horn, Violin, ...       0.99  Classical             0.0
3502  Die Zauberflöte, K.620: "D...       0.99      Opera             0.0

[3503 rows x 4 columns]


In [14]:
# Problem 7: Find the most expensive tracks in each genre


query = """
           SELECT 
               Name,
               Genre,
               UnitPrice,
               PriceRank
            FROM (
                SELECT
                    Track.Name AS Name,
                    Genre.Name AS Genre,
                    Track.UnitPrice AS UnitPrice,
                    RANK() OVER (
                        PARTITION BY Genre.GenreId
                        ORDER BY Track.UnitPrice DESC
                    ) AS PriceRank
                FROM Track
                JOIN Genre ON Track.GenreId = Genre.GenreId
            ) AS ranked
            WHERE PriceRank = 1;
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                               Name      Genre  UnitPrice  PriceRank
0     For Those About To Rock (W...       Rock       0.99          1
1                 Balls to the Wall       Rock       0.99          1
2                   Fast As a Shark       Rock       0.99          1
3                 Restless and Wild       Rock       0.99          1
4              Princess of the Dawn       Rock       0.99          1
...                             ...        ...        ...        ...
3498  Pini Di Roma (Pinien Von R...  Classical       0.99          1
3499  String Quartet No. 12 in C...  Classical       0.99          1
3500  L'orfeo, Act 3, Sinfonia (...  Classical       0.99          1
3501  Quintet for Horn, Violin, ...  Classical       0.99          1
3502  Die Zauberflöte, K.620: "D...      Opera       0.99          1

[3503 rows x 4 columns]


In [16]:
# Problem 8: Find the most expensive tracks in each genre


query = """
           WITH RankedTracks AS (
               SELECT
                   Track.Name AS Name,
                   Genre.Name AS Genre,
                   Track.UnitPrice AS UnitPrice,
                   ROW_NUMBER() OVER(
                       PARTITION BY Genre.GenreId
                       ORDER BY Track.UnitPrice DESC
                   ) AS TrackRank
                FROM Track
                JOIN Genre ON Track.GenreId = Genre.GenreId
           )

           SELECT Name, Genre, UnitPrice
           FROM RankedTracks
           WHERE TrackRank = 1;
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                             Name               Genre  UnitPrice
0   For Those About To Rock (W...                Rock       0.99
1                      Desafinado                Jazz       0.99
2                   Enter Sandman               Metal       0.99
3              Your Time Has Come  Alternative & Punk       0.99
4                           Money       Rock And Roll       0.99
5      First Time I Met The Blues               Blues       0.99
6              Jorge Da Capadócia               Latin       0.99
7                        Girassol              Reggae       0.99
8   Dig-Dig, Lambe-Lambe (Ao V...                 Pop       0.99
9                    Vai-Vai 2001          Soundtrack       0.99
10                Samba Da Bênção          Bossa Nova       0.99
11                         My Way      Easy Listening       0.99
12                 Wildest Dreams         Heavy Metal       0.99
13           Please Please Please            R&B/Soul       0.99
14             Just Anoth

In [18]:
# Problem 9: Show each customer's most recent invoice date and amount


query = """
        With RankedCustomers AS (
            SELECT
                Customer.FirstName AS Name,
                Invoice.Total AS Most_Recent_Invoice,
                Invoice.InvoiceDate AS Date,
                ROW_NUMBER() OVER(
                    PARTITION BY Customer.CustomerId
                    ORDER BY Invoice.InvoiceDate DESC
                ) AS CustomerRank
            FROM Customer
            JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
                
        )

        SELECT Name, Most_Recent_Invoice, Date
        FROM RankedCustomers
        WHERE CustomerRank = 1;

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

         Name  Most_Recent_Invoice                 Date
0        Luís                 8.91  2025-08-07 00:00:00
1      Leonie                 0.99  2024-07-13 00:00:00
2    François                 0.99  2025-09-20 00:00:00
3       Bjørn                 1.98  2025-10-03 00:00:00
4   František                 8.91  2025-05-06 00:00:00
5      Helena                25.86  2025-11-13 00:00:00
6      Astrid                 0.99  2025-06-19 00:00:00
7        Daan                 3.96  2025-10-04 00:00:00
8        Kara                 8.91  2025-02-02 00:00:00
9     Eduardo                13.86  2025-08-12 00:00:00
10  Alexandre                 0.99  2025-03-18 00:00:00
11    Roberto                 5.94  2025-10-05 00:00:00
12   Fernanda                 8.91  2024-11-01 00:00:00
13       Mark                13.86  2025-05-11 00:00:00
14   Jennifer                 0.99  2024-12-15 00:00:00
15      Frank                 5.94  2025-07-04 00:00:00
16       Jack                10.91  2024-07-31 0

In [25]:
# Problem 10: For each artist, show their 2nd most popular album (by track count)


query = """
        WITH RankedAlbums AS (
            SELECT
                Album.Title AS Album,
                COUNT(Track.TrackId) AS Number_Of_Tracks,
                Artist.Name AS Artist,
                DENSE_RANK() OVER (
                    PARTITION BY Artist.ArtistId
                    ORDER BY COUNT(Track.TrackId) DESC
                ) AS TrackRank
            FROM Artist
            JOIN Album ON Artist.ArtistId = Album.ArtistId
            JOIN Track ON Album.AlbumId = Track.AlbumId
            GROUP BY Artist.ArtistId, Artist.Name, Album.AlbumId, Album.Title
        )

        SELECT Artist, Number_Of_Tracks 
        FROM RankedAlbums
        WHERE TrackRank = 2;
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                         Artist  Number_Of_Tracks
0                         AC/DC                 8
1                        Accept                 1
2          Antônio Carlos Jobim                14
3                    Audioslave                12
4           Black Label Society                 5
5                 Black Sabbath                 7
6                Caetano Veloso                 3
7   Chico Science & Nação Zumbi                13
8                  Cidade Negra                14
9                  Led Zeppelin                10
10                 Led Zeppelin                10
11                 Gilberto Gil                14
12                    Metallica                14
13                        Queen                11
14                         Kiss                15
15                   Spyro Gyra                 9
16                    Green Day                13
17                  Deep Purple                11
18                      Santana                 8


In [28]:
# Problem 11: Calculate 7-Day Moving average of daily revenue.


query = """
            SELECT
                Invoice.InvoiceDate AS Date,
                SUM(Invoice.Total) AS Daily_Revenue,
                AVG(SUM(Invoice.Total)) OVER(
                    ORDER BY Invoice.InvoiceDate
                    ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
                ) AS MOVING7DAYAVG
            FROM Invoice
            GROUP BY Invoice.InvoiceDate;
        
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                    Date  Daily_Revenue  MOVING7DAYAVG
0    2021-01-01 00:00:00           1.98       1.980000
1    2021-01-02 00:00:00           3.96       2.970000
2    2021-01-03 00:00:00           5.94       3.960000
3    2021-01-06 00:00:00           8.91       5.197500
4    2021-01-11 00:00:00          13.86       6.930000
..                   ...            ...            ...
349  2025-12-05 00:00:00           3.96       7.654286
350  2025-12-06 00:00:00           5.94       7.937143
351  2025-12-09 00:00:00           8.91       8.361429
352  2025-12-14 00:00:00          13.86       9.068571
353  2025-12-22 00:00:00           1.99       5.658571

[354 rows x 3 columns]


In [3]:
# Problem 12: Show each invoice with revenue from previous invoice for same customer


query = """
           SELECT Customer.FirstName AS Name,
           Invoice.InvoiceDate,
           Invoice.Total as CurrentInvoice,
           LAG(Invoice.Total, 1) OVER(PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS PreviousInvoice,
           LEAD(Invoice.Total, 1) OVER(PARTITION BY Customer.CustomerId ORDER BY Invoice.InvoiceDate) AS NextInvoice
           FROM Customer
           JOIN Invoice ON Customer.CustomerId = Invoice.CustomerId
           ORDER BY Invoice.InvoiceDate;
        
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

         Name          InvoiceDate  CurrentInvoice  PreviousInvoice  \
0      Leonie  2021-01-01 00:00:00            1.98              NaN   
1       Bjørn  2021-01-02 00:00:00            3.96              NaN   
2        Daan  2021-01-03 00:00:00            5.94              NaN   
3        Mark  2021-01-06 00:00:00            8.91              NaN   
4        John  2021-01-11 00:00:00           13.86              NaN   
..        ...                  ...             ...              ...   
407    Victor  2025-12-05 00:00:00            3.96             1.98   
408    Robert  2025-12-06 00:00:00            5.94             3.96   
409  Madalena  2025-12-09 00:00:00            8.91            13.86   
410     Terhi  2025-12-14 00:00:00           13.86             1.98   
411     Manoj  2025-12-22 00:00:00            1.99             5.94   

     NextInvoice  
0          13.86  
1           5.94  
2           0.99  
3           1.98  
4           8.91  
..           ...  
407          N

In [6]:
# Problem 13: Calculate month-over month revenue growth percentage 


query = """
        WITH MonthlyRevenue AS (
            SELECT
                strftime('%Y-%m', InvoiceDate) AS Month,
                SUM(Total) AS MonthlyRevenue
            FROM Invoice
            GROUP BY strftime('%Y-%m', InvoiceDate)
        )
        SELECT
            Month,
            MonthlyRevenue,
            LAG(MonthlyRevenue, 1) OVER (ORDER BY Month) AS PreviousMonthRevenue,
            ROUND(
                ((MonthlyRevenue - LAG(MonthlyRevenue, 1) OVER (ORDER By Month))
                / LAG(MonthlyRevenue, 1) OVER (ORDER BY Month)) * 100,
                2
            ) AS GrowthPercentage
            FROM MonthlyRevenue
            ORDER BY Month;
        
            
           
        
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

      Month  MonthlyRevenue  PreviousMonthRevenue  GrowthPercentage
0   2021-01           35.64                   NaN               NaN
1   2021-02           37.62                 35.64              5.56
2   2021-03           37.62                 37.62              0.00
3   2021-04           37.62                 37.62              0.00
4   2021-05           37.62                 37.62              0.00
5   2021-06           37.62                 37.62              0.00
6   2021-07           37.62                 37.62              0.00
7   2021-08           37.62                 37.62              0.00
8   2021-09           37.62                 37.62              0.00
9   2021-10           37.62                 37.62              0.00
10  2021-11           37.62                 37.62              0.00
11  2021-12           37.62                 37.62              0.00
12  2022-01           52.62                 37.62             39.87
13  2022-02           46.62                 52.6

In [13]:
# Problem 14: Assign each track to a price quartile within its genre

query = """
        SELECT
            Track.Name AS Track,
            Track.UnitPrice AS Price,
                NTILE(4) OVER (PARTITION BY Genre.GenreId ORDER BY Track.UnitPrice) AS PriceQuartile
            FROM Track
            JOIN Genre ON Track.GenreId = Genre.GenreId
            WHERE Genre.Name = 'TV Shows';
            
        
            
           
        
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)


                     Track  Price  PriceQuartile
0   Occupation / Precipice   1.99              1
1            Exodus, Pt. 1   1.99              1
2            Exodus, Pt. 2   1.99              1
3            Collaborators   1.99              1
4                     Torn   1.99              1
..                     ...    ...            ...
88              The Merger   1.99              4
89                   Pilot   1.99              4
90                 Eggtown   1.99              4
91                 Ji Yeon   1.99              4
92      Meet Kevin Johnson   1.99              4

[93 rows x 3 columns]


In [18]:
# Problem 15: Show tracks where length is in top 10% of their Genre

query = """
        WITH RankedTracks AS (
        SELECT
            Track.Name AS Track,
            Track.Milliseconds AS Length,
            Genre.Name AS Genre,
                NTILE(10) OVER (PARTITION BY Genre.GenreId ORDER BY Track.Milliseconds DESC) AS TrackLength
            FROM Track
            JOIN Genre ON Track.GenreId = Genre.GenreId
        )

        SELECT Track, Length, Genre
        FROM RankedTracks
        WHERE TrackLength = 1;
            
            
        
            
           
        
       

        
           
        
        """

result = pd.read_sql_query(query, conn)
print(result)

                             Track   Length      Genre
0               Dazed And Confused  1612329       Rock
1                   Space Truckin'  1196094       Rock
2               Dazed And Confused  1116734       Rock
3    We've Got To Get Together/...  1070027       Rock
4                      Funky Piano   934791       Rock
..                             ...      ...        ...
357  Concerto for Piano No. 2 i...   560342  Classical
358  Scheherazade, Op. 35: I. T...   545203  Classical
359   On the Beautiful Blue Danube   526696  Classical
360  Jupiter, the Bringer of Jo...   522099  Classical
361  Die Zauberflöte, K.620: "D...   174813      Opera

[362 rows x 3 columns]
